<a href="https://colab.research.google.com/github/cdxing/applied-scientist-training/blob/main/05_long_term_recommendation/phlrec_toy_reproduction_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Phase 0 — Dataset Design

### Unit of observation
One row represents one customer at one decision point.

### Inputs
- Customer features
- Item features
- Current context

### Multi-horizon targets
- 1-day retention
- 7-day retention
- 14-day retention
- 30-day retention
- 60-day retention

### Masks
A horizon contributes to the loss only after its label has matured.

### Scientific requirement
Short-term outcomes should be correlated with, but not perfectly predict,
long-term retention.

This allows us to test whether progressive horizon learning improves
long-term optimization without waiting for all long-horizon labels.

## Simulation Assumptions

### Customer state
What variables describe a customer at time t?

### State evolution
How does customer behavior change over time?

### Churn mechanism
What factors increase or decrease churn probability?

### Billing event
How does a renewal/charge event affect low-engagement customers?

### Action effect
How can a recommendation or intervention change future behavior?

### Multi-horizon labels
How are 1D, 7D, 14D, 30D, and 60D retention labels generated?

## Phase 1 — Simulation Specification

### Scientific Question
Can current and recent customer behavior provide useful signals for
future Prime membership retention across multiple time horizons?

### Unit of Observation
One customer at one decision point.

### Observed State
- tenure_months
- orders_last_30d
- video_hours_last_30d
- days_since_last_order
- activity_drop_30d
- support_contacts_30d
- billing_event

### Latent State
Customer engagement / perceived membership value.

### State Evolution
Customer behavior changes over time through stochastic activity,
engagement drift, and periodic billing events.

### Churn Mechanism
Low engagement, inactivity, activity decline, and billing events under
low perceived value increase churn probability.

### Outputs
Retention labels at:
1D, 7D, 14D, 30D, and 60D.

### Important Assumption
This is a synthetic world designed to test modeling ideas.
The assumed relationships are hypotheses, not claims about real Amazon customers.

## Phase 2 — Customer Event Simulator

### Raw Event Log

The simulator generates event-level customer histories.

Each row represents one observed event rather than one customer snapshot.

Initial event types:

- purchase
- video_watch
- support_contact
- billing_event
- churn

### Latent Customer State

Each customer has an unobserved engagement state that evolves over time.

Higher engagement should generally produce more purchasing and service usage,
while lower engagement should increase inactivity and churn propensity.

### Data Pipeline

latent state
→ observed events
→ feature engineering
→ customer snapshot at decision time
→ future retention labels

### Important Constraint

Features must only use information available before the decision point.
Future behavior is used only to construct labels.

## Session 1 — Generate Initial Customer Population

### What
Generate 1,000 heterogeneous Prime-like customers at Day 0.

### Why
Customers should not all behave the same way. We need variation in engagement,
purchase behavior, video usage, long-term trend, and membership tenure before
simulating their future trajectories.

### How
Sample each customer's initial characteristics from distributions chosen to
match the variable's valid range and qualitative behavior.

### What to notice
These distributions are modeling assumptions, not claims about real Amazon users.
Later we can test sensitivity to these assumptions or replace them with real data.

In [1]:
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

N_CUSTOMERS = 1000

df_customers = pd.DataFrame({
    "user_id": np.arange(N_CUSTOMERS),

    # [0, 1]: most customers are around the middle,
    # with fewer extremely disengaged or highly engaged users.
    "initial_engagement": rng.beta(
        a=3,
        b=3,
        size=N_CUSTOMERS
    ),

    # Persistent tendency to gain or lose engagement each day.
    # Can be positive or negative.
    "engagement_drift": rng.normal(
        loc=0.0,
        scale=0.003,
        size=N_CUSTOMERS
    ),

    # Baseline expected purchases per day.
    # Positive and right-skewed.
    "baseline_purchase_rate": rng.gamma(
        shape=2.0,
        scale=0.08,
        size=N_CUSTOMERS
    ),

    # Baseline Prime-like video usage in minutes per day.
    # Positive and right-skewed.
    "baseline_video_minutes": rng.gamma(
        shape=2.0,
        scale=10.0,
        size=N_CUSTOMERS
    ),

    # Membership tenure: many relatively new users,
    # fewer very long-term users.
    "tenure_months": np.clip(
        rng.exponential(scale=18, size=N_CUSTOMERS) + 1,
        1,
        120
    ).astype(int)
})

In [2]:
df_customers.head()

,user_id,initial_engagement,engagement_drift,baseline_purchase_rate,baseline_video_minutes,tenure_months
0,0,0.438671,0.002061,0.034227,10.055207,16
1,1,0.521883,0.005168,0.133269,3.828039,17
2,2,0.611857,0.000513,0.094552,44.290616,6
3,3,0.513927,-0.000339,0.087258,19.341187,9
4,4,0.647985,0.004526,0.246965,24.099721,37


In [3]:
df_customers.describe()

,user_id,initial_engagement,engagement_drift,baseline_purchase_rate,baseline_video_minutes,tenure_months
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,499.500000,0.505423,-0.000095,0.165440,19.560249,18.370000
std,288.819436,0.186466,0.002992,0.112625,13.290959,17.527873
min,0.000000,0.062508,-0.009431,0.003644,0.451580,1.000000
25%,249.750000,0.364561,-0.002081,0.083742,9.702643,5.000000
50%,499.500000,0.503703,-0.000083,0.136600,16.640704,13.000000
75%,749.250000,0.642776,0.001739,0.223810,26.485095,26.000000
max,999.000000,0.968601,0.010362,0.681991,83.478834,106.000000


In [4]:
print("Engagement range:",
      df_customers["initial_engagement"].min(),
      df_customers["initial_engagement"].max())

print("Mean purchase rate/day:",
      df_customers["baseline_purchase_rate"].mean())

print("Mean video minutes/day:",
      df_customers["baseline_video_minutes"].mean())

print("Median tenure months:",
      df_customers["tenure_months"].median())

Engagement range: 0.06250804555927479 0.9686007405669117
Mean purchase rate/day: 0.16543959933201888
Mean video minutes/day: 19.560249467124233
Median tenure months: 13.0
